<a href="https://colab.research.google.com/github/JiHyeonKu/JeonGiPeu/blob/main/conv15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import sys
import os
import calendar
from datetime import datetime, timedelta
from collections import Counter

# ── Windows 터미널 UTF-8 강제 적용 (§2.1) ────────────────────
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
    sys.stdin.reconfigure(encoding="utf-8")

# ── 파일 경로 상수 ────────────────────────────────────────────
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

PRODUCT_DATA = os.path.join(BASE_DIR, "productData.txt")
PRODUCT_LOG = os.path.join(BASE_DIR, "productLog.txt")
CUSTOMER_DATA = os.path.join(BASE_DIR, "customerData.txt")
CUSTOMER_LOG = os.path.join(BASE_DIR, "customerLog.txt")
CATEGORY_DATA = os.path.join(BASE_DIR, "categoryData.txt")
COUPON_DATA = os.path.join(BASE_DIR, "couponData.txt")

FILES = [PRODUCT_DATA, PRODUCT_LOG, CUSTOMER_DATA, CUSTOMER_LOG, CATEGORY_DATA, COUPON_DATA]

# 프로그램 전역 상태
current_datetime: datetime | None = None  # 6.1.4 현재 일시

# ══════════════════════════════════════════════════════════════
#  SECTION 1 : 파일 I/O 헬퍼
# ══════════════════════════════════════════════════════════════
def read_file(path: str) -> list[str]:
    if not os.path.exists(path):
        print(f"{path} 파일이 존재하지 않습니다.")
        sys.exit(1)
    with open(path, "r", encoding="utf-8", newline="") as f:
        return [line.rstrip("\r\n") for line in f if line.strip()]

def write_file(path: str, lines: list[str]) -> None:
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

def append_line(path: str, line: str) -> None:
    with open(path, "a+", encoding="utf-8") as f:
        f.seek(0, os.SEEK_END)
        if f.tell() > 0:
            f.seek(f.tell() - 1)
            if f.read(1) != "\n":
                f.write("\n")
        f.write(line + "\n")

def parse_record(line: str) -> list[str]:
    return [field.strip() for field in line.split("|")]

# ══════════════════════════════════════════════════════════════
#  SECTION 2 : 유효성 검사 (4절 데이터 요소)
# ══════════════════════════════════════════════════════════════
DATETIME_RE = re.compile(
    r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])"
    r"\s([01]\d|2[0-3]):[0-5]\d:[0-5]\d$"
)
PAYMENT_METHODS = {"현금", "카드", "포인트"}
EVENT_TYPES = {"IN", "OUT", "DISPOSE"}
CUSTOMER_NAME_RE = re.compile(r"^([가-힣]{2,5}|[a-zA-Z]{3,20})$")
EVENT_ID_RE = re.compile(r"^[1-9][0-9]*$")
PURCHASE_ID_RE = re.compile(r"^[1-9][0-9]*$")
PRODUCT_ID_RE = re.compile(r"^\d{4}$")
CUSTOMER_ID_RE = re.compile(r"^\d{4}$")
CATEGORY_ID_RE = re.compile(r"^C(00[1-9]|0[1-9][0-9]|[1-9][0-9]{2})$")
COUPON_ID_RE = re.compile(r"^[1-9][0-9]*$")
GRADE_RE = re.compile(r"^(BRONZE|SILVER|GOLD|VIP)$")
COUPON_TYPE_RE = re.compile(r"^(RATE|FLAT)$")
COUPON_STATUS_RE = re.compile(r"^(USABLE|USED)$")

def is_valid_datetime(s: str) -> bool:
    if not DATETIME_RE.match(s):
        return False
    try:
        datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
        return True
    except ValueError:
        return False

def parse_dt(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def is_valid_product_id(s: str) -> bool:
    return bool(PRODUCT_ID_RE.match(s))

def validate_event_time(pid_list, event_time):
    products = _load_products()
    if current_datetime is None:
        print("현재 일시가 설정되지 않았습니다.")
        return False
    if event_time > current_datetime:
        print("이벤트 시각은 현재보다 미래일 수 없습니다.")
        return False
    for pid in pid_list:
        if pid not in products:
            print("존재하지 않는 상품입니다.")
            return False
        in_date = parse_dt(products[pid]["in_date"])
        if event_time < in_date:
            print("이벤트 시각은 입고일보다 이전일 수 없습니다.")
            return False
    return True

def is_valid_customer_id(s: str) -> bool:
    return bool(CUSTOMER_ID_RE.match(s))

def is_valid_category(s: str) -> bool:
    category_names = {parse_record(line)[1] for line in read_file(CATEGORY_DATA) if len(parse_record(line)) == 2}
    return s in category_names

def is_valid_price(s: str) -> bool:
    return s.isdigit()

def is_valid_customer_name(s: str) -> bool:
    return bool(CUSTOMER_NAME_RE.match(s))

def is_valid_event_id(s: str) -> bool:
    return bool(EVENT_ID_RE.match(s))

# ══════════════════════════════════════════════════════════════
#  SECTION 3 : 재고 계산 (이벤트 로그 기반)
# ══════════════════════════════════════════════════════════════
def calc_stock(product_id: str, as_of: datetime | None = None) -> int:
    stock = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        if len(fields) != 4:
            continue
        event_id, ids_raw, etype, evt_time_s = fields
        ids = [i.strip() for i in ids_raw.split(",") if i.strip()]
        if product_id not in ids:
            continue
        if as_of and parse_dt(evt_time_s) > as_of:
            continue
        if etype == "IN":
            stock += ids.count(product_id)
        elif etype in ("OUT", "DISPOSE"):
            stock -= ids.count(product_id)
    return stock

def get_all_stocks(as_of: datetime | None = None) -> dict[str, int]:
    product_ids = {parse_record(l)[0] for l in read_file(PRODUCT_DATA)}
    return {pid: calc_stock(pid, as_of) for pid in product_ids}

def next_available_4digit_id(used_ids: set[str]) -> str | None:
    for num in range(1, 10000):
        candidate = str(num).zfill(4)
        if candidate not in used_ids:
            return candidate
    return None

def next_event_id() -> str:
    max_id = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        if fields and EVENT_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_purchase_id() -> str:
    max_id = 0
    for line in read_file(CUSTOMER_LOG):
        fields = parse_record(line)
        if fields and PURCHASE_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_product_id() -> str | None:
    used_ids = set()
    for line in read_file(PRODUCT_DATA):
        fields = parse_record(line)
        if fields and PRODUCT_ID_RE.match(fields[0]):
            used_ids.add(fields[0])
    return next_available_4digit_id(used_ids)

def next_customer_id() -> str | None:
    used_ids = set()
    for line in read_file(CUSTOMER_DATA):
        fields = parse_record(line)
        if fields and CUSTOMER_ID_RE.match(fields[0]) and fields[0] != "0000":
            used_ids.add(fields[0])
    return next_available_4digit_id(used_ids)

def get_last_reference_time() -> datetime | None:
    times = []
    for path in (PRODUCT_LOG, CUSTOMER_LOG):
        for line in read_file(path):
            fields = parse_record(line)
            if len(fields) >= 4 and is_valid_datetime(fields[-1]):
                times.append(parse_dt(fields[-1]))

    for line in read_file(CUSTOMER_DATA):
        fields = parse_record(line)
        if len(fields) >= 5 and is_valid_datetime(fields[4]):
            times.append(parse_dt(fields[4]))

    return max(times) if times else None

# ══════════════════════════════════════════════════════════════
#  SECTION 4 : 무결성 검사 (2.6절, 5.9, 5.10절)
# ══════════════════════════════════════════════════════════════
def ids_key(ids: list[str]) -> tuple[str, ...]:
    return tuple(sorted(pid.strip() for pid in ids))

def check_integrity() -> None:
    print("무결성 검사 중… ", end="", flush=True)
    errors = []

    # ── categoryData.txt 검사 ─────────────────────────────
    category_ids = set()
    category_names = set()
    for line in read_file(CATEGORY_DATA):
        f = parse_record(line)
        if len(f) != 2:
            errors.append(f"<{CATEGORY_DATA}> => {line}")
            continue
        cid, cname = f[0], f[1]
        ok = (
            CATEGORY_ID_RE.match(cid)
            and 1 <= len(cname) <= 10
            and " " not in cname
            and "|" not in cname and "\\" not in cname and "/" not in cname
        )
        if not ok:
            errors.append(f"<{CATEGORY_DATA}> => {line}")
        elif cid in category_ids or cname in category_names:
            errors.append(f"<{CATEGORY_DATA}> 중복 카테고리 => {line}")
        else:
            category_ids.add(cid)
            category_names.add(cname)

    for req_cat in [f"C{str(i).zfill(3)}" for i in range(1, 8)]:
        if req_cat not in category_ids:
            errors.append(f"<{CATEGORY_DATA}> 기본 카테고리 누락 => {req_cat}")

    # ── productData.txt 검사 ─────────────────────────────
    product_ids = set()
    products = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        ok = (
                len(f) == 6
                and is_valid_product_id(f[0])
                and len(f[1]) >= 1
                and "\\" not in f[1]
                and "/" not in f[1]
                and "|" not in f[1]
                and is_valid_price(f[2])
                and f[3] in category_names
                and is_valid_datetime(f[4])
                and is_valid_datetime(f[5])
                and parse_dt(f[4]) <= parse_dt(f[5])
        )
        if not ok:
            errors.append(f"<{PRODUCT_DATA}> => {line}")
        else:
            if f[0] in product_ids:
                errors.append(f"<{PRODUCT_DATA}> 중복 상품ID => {line}")
            product_ids.add(f[0])
            products[f[0]] = {
                "name": f[1], "price": f[2], "category": f[3],
                "in_date": f[4], "exp_date": f[5],
            }

    # ── productLog.txt 검사 ─────────────────────────────
    product_events = []
    event_ids = set()
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        ids = []
        ok = (
                len(f) == 4
                and is_valid_event_id(f[0])
                and f[2] in EVENT_TYPES
                and is_valid_datetime(f[3])
        )
        if ok:
            ids = [i.strip() for i in f[1].split(",") if i.strip()]
            if not ids:
                ok = False
            if any(pid not in product_ids for pid in ids):
                ok = False
            if len(set(ids)) != len(ids):
                ok = False
            if ok:
                event_time = parse_dt(f[3])
                for pid in ids:
                    if event_time < parse_dt(products[pid]["in_date"]):
                        ok = False
                        break
        if not ok:
            errors.append(f"<{PRODUCT_LOG}> => {line}")
        elif f[0] in event_ids:
            errors.append(f"<{PRODUCT_LOG}> 중복 이벤트ID => {line}")
        else:
            event_ids.add(f[0])
            product_events.append({
                "event_id": f[0], "ids": ids, "etype": f[2],
                "qty": len(ids), "time": f[3], "line": line,
            })

    # ── couponData.txt 검사 ─────────────────────────────
    coupon_ids = set()
    coupon_map = {}
    coupon_owner = {}
    for line in read_file(COUPON_DATA):
        f = parse_record(line)
        if len(f) != 5:
            errors.append(f"<{COUPON_DATA}> => {line}")
            continue
        cid, ctype, cval, cdur, cstat = f[0], f[1], f[2], f[3], f[4]

        ok = True
        if not COUPON_ID_RE.match(cid): ok = False
        if not COUPON_TYPE_RE.match(ctype): ok = False
        if not is_valid_datetime(cdur): ok = False
        if not COUPON_STATUS_RE.match(cstat): ok = False

        if ctype == "RATE" and not (cval.isdigit() and 1 <= int(cval) <= 100): ok = False
        if ctype == "FLAT" and not (cval.isdigit() and int(cval) >= 1): ok = False

        if not ok:
            errors.append(f"<{COUPON_DATA}> => {line}")
        elif cid in coupon_ids:
            errors.append(f"<{COUPON_DATA}> 중복 쿠폰ID => {line}")
        else:
            coupon_ids.add(cid)
            coupon_map[cid] = {"type": ctype, "val": cval, "dur": cdur, "stat": cstat}

    # ── customerData.txt 검사 ─────────────────────────────
    customer_ids = set()
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        ok = (
                len(f) == 6
                and is_valid_customer_id(f[0])
                and f[0] != "0000"
                and is_valid_customer_name(f[1])
                and f[2].isdigit()
                and GRADE_RE.match(f[3])
                and is_valid_datetime(f[4])
        )
        if not ok:
            errors.append(f"<{CUSTOMER_DATA}> => {line}")
        elif f[0] in customer_ids:
            errors.append(f"<{CUSTOMER_DATA}> 중복 고객ID => {line}")
        else:
            customer_ids.add(f[0])

            coupon_str = f[5]
            if coupon_str:
                c_list = [c.strip() for c in coupon_str.split(",")]
                if len(set(c_list)) != len(c_list):
                    errors.append(f"<{CUSTOMER_DATA}> 쿠폰ID 중복 보유 => {line}")
                for c in c_list:
                    if not COUPON_ID_RE.match(c):
                        errors.append(f"<{CUSTOMER_DATA}> 쿠폰ID 형식 오류 => {line}")
                    elif c not in coupon_ids:
                        errors.append(f"<{CUSTOMER_DATA}> 존재하지 않는 쿠폰 참조 => {line}")
                    elif coupon_map[c]["stat"] != "USABLE":
                        errors.append(f"<{CUSTOMER_DATA}> USED 쿠폰 보유 => {line}")
                    elif c in coupon_owner:
                        errors.append(f"<{CUSTOMER_DATA}> 쿠폰 소유 중복 => {line}")
                    else:
                        coupon_owner[c] = f[0]

    for cid, cmap in coupon_map.items():
        if cmap["stat"] == "USABLE" and cid not in coupon_owner:
            errors.append(f"<{COUPON_DATA}> 소유자 없는 USABLE 쿠폰 => {cid}")

    # ── customerLog.txt 검사 ─────────────────────────────
    purchase_ids = set()
    purchase_records = []
    used_coupon_ids_in_log = set()
    for line in read_file(CUSTOMER_LOG):
        f = parse_record(line)
        ids = []
        ok = (
                len(f) == 8
                and PURCHASE_ID_RE.match(f[0])
                and is_valid_customer_id(f[1])
                and (f[1] == "0000" or f[1] in customer_ids)
                and is_valid_price(f[3])
                and (f[4] == "" or f[4] in coupon_ids)
                and f[5] in PAYMENT_METHODS
                and f[6].isdigit() and int(f[6]) >= 1
                and is_valid_datetime(f[7])
        )
        if ok:
            ids = [i.strip() for i in f[2].split(",") if i.strip()]
            if not ids:
                ok = False
            if any(pid not in product_ids for pid in ids):
                ok = False
            if int(f[6]) != len(ids):
                ok = False
            if len(set(ids)) != len(ids):
                ok = False
            if f[1] == "0000" and f[5] == "포인트":
                ok = False
            if f[4] != "" and f[5] == "포인트":
                ok = False
            if f[4] != "":
                if f[4] not in coupon_map or coupon_map[f[4]]["stat"] != "USED":
                    ok = False
                else:
                    used_coupon_ids_in_log.add(f[4])

        if not ok:
            errors.append(f"<{CUSTOMER_LOG}> => {line}")
        elif f[0] in purchase_ids:
            errors.append(f"<{CUSTOMER_LOG}> 중복 구매ID => {line}")
        else:
            purchase_ids.add(f[0])
            purchase_records.append({
                "purchase_id": f[0], "customer_id": f[1], "ids": ids,
                "price": int(f[3]), "coupon_id": f[4], "payment": f[5],
                "qty": int(f[6]), "time": f[7], "line": line,
            })

    for cid, cmap in coupon_map.items():
        if cmap["stat"] == "USED" and cid not in used_coupon_ids_in_log:
            errors.append(f"<{COUPON_DATA}> customerLog에 사용 기록이 없는 USED 쿠폰 => {cid}")

    out_counter = Counter()
    out_line_map = {}
    for ev in product_events:
        if ev["etype"] == "OUT":
            key = (ids_key(ev["ids"]), ev["qty"], ev["time"])
            out_counter[key] += 1
            out_line_map.setdefault(key, ev["line"])

    purchase_counter = Counter()
    purchase_line_map = {}
    for pr in purchase_records:
        key = (ids_key(pr["ids"]), pr["qty"], pr["time"])
        purchase_counter[key] += 1
        purchase_line_map.setdefault(key, pr["line"])

    for key, count in out_counter.items():
        if purchase_counter[key] < count:
            errors.append(f"<{PRODUCT_LOG}> OUT 이벤트에 대응되는 구매 기록 불일치 => {out_line_map[key]}")

    for key, count in purchase_counter.items():
        if out_counter[key] < count:
            errors.append(f"<{CUSTOMER_LOG}> 구매 기록에 대응되는 OUT 이벤트 불일치 => {purchase_line_map[key]}")

    stock_by_pid = {pid: 0 for pid in product_ids}
    sorted_events = sorted(
        product_events,
        key=lambda ev: (parse_dt(ev["time"]), int(ev["event_id"]))
    )

    for ev in sorted_events:
        id_counts = Counter(ev["ids"])
        if ev["etype"] == "IN":
            for pid, count in id_counts.items():
                stock_by_pid[pid] += count
        elif ev["etype"] in ("OUT", "DISPOSE"):
            flow_ok = True
            for pid, count in id_counts.items():
                if stock_by_pid.get(pid, 0) < count:
                    errors.append(f"<{PRODUCT_LOG}> 재고 흐름 불일치 => {ev['line']}")
                    flow_ok = False
                    break
            if flow_ok:
                for pid, count in id_counts.items():
                    stock_by_pid[pid] -= count

    if errors:
        print("데이터 오류가 발생했습니다.")
        for e in errors:
            print(e)
        sys.exit(1)
    else:
        print("완료")

# ══════════════════════════════════════════════════════════════
#  SECTION 5 : 카테고리 관리 함수 (2.1절)
# ══════════════════════════════════════════════════════════════
def manage_category_menu() -> None:
    while True:
        print("\n1. 카테고리 추가\n2. 카테고리 수정\n3. 카테고리 삭제\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()

        if not re.fullmatch(r"[0-9]+", choice):
            print("0 이상의 정수를 입력해주세요.")  # 피드백 반영
            continue

        match choice:
            case "1":
                add_category()
            case "2":
                modify_category()
            case "3":
                delete_category()
            case "0":
                return
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

def add_category() -> None:
    c_name = input("추가할 카테고리명: ").strip()
    if len(c_name) < 1 or len(c_name) > 10 or " " in c_name or "|" in c_name or "\\" in c_name or "/" in c_name:
        print("올바른 카테고리 명칭을 입력해 주세요.")
        return

    lines = read_file(CATEGORY_DATA)
    max_id = 0
    for line in lines:
        f = parse_record(line)
        if len(f) == 2:
            if f[1] == c_name:
                print("이미 존재하는 카테고리입니다.")
                return
            if f[0].startswith("C") and f[0][1:].isdigit():
                max_id = max(max_id, int(f[0][1:]))

    if max_id >= 999:
        print("더 이상 카테고리를 추가할 수 없습니다.")
        return

    new_cid = f"C{str(max_id + 1).zfill(3)}"
    append_line(CATEGORY_DATA, f"{new_cid}|{c_name}")
    print("카테고리가 성공적으로 추가되었습니다.")

def modify_category() -> None:
    lines = read_file(CATEGORY_DATA)
    print("\n[현재 등록된 카테고리]")
    cmap = {}
    for line in lines:
        f = parse_record(line)
        if len(f) == 2:
            print(f"{f[0]} - {f[1]}")
            cmap[f[0]] = f[1]

    cid = input("\n수정할 카테고리ID: ").strip()
    if cid not in cmap:
        print("존재하지 않는 카테고리ID입니다.")
        return

    if cid in [f"C{str(i).zfill(3)}" for i in range(1, 8)]:
        print("기본 카테고리는 수정할 수 없습니다.")
        return

    new_name = input("변경할 새 카테고리명: ").strip()
    if len(new_name) < 1 or len(new_name) > 10 or " " in new_name or "|" in new_name or "\\" in new_name or "/" in new_name:
        print("올바른 카테고리 명칭을 입력해 주세요.")
        return

    if new_name in cmap.values():
        print("이미 존재하는 카테고리입니다.")
        return

    old_name = cmap[cid]

    updated_cat = []
    for line in lines:
        f = parse_record(line)
        if f[0] == cid:
            updated_cat.append(f"{cid}|{new_name}")
        else:
            updated_cat.append(line)
    write_file(CATEGORY_DATA, updated_cat)

    prod_lines = read_file(PRODUCT_DATA)
    updated_prod = []
    for line in prod_lines:
        f = parse_record(line)
        if len(f) == 6 and f[3] == old_name:
            f[3] = new_name
            updated_prod.append("|".join(f))
        else:
            updated_prod.append(line)
    write_file(PRODUCT_DATA, updated_prod)

    print("카테고리 명칭이 성공적으로 수정되었습니다.")

def delete_category() -> None:
    lines = read_file(CATEGORY_DATA)
    print("\n[현재 등록된 카테고리]")
    cmap = {}
    for line in lines:
        f = parse_record(line)
        if len(f) == 2:
            print(f"{f[0]} - {f[1]}")
            cmap[f[0]] = f[1]

    cid = input("\n삭제할 카테고리ID: ").strip()
    if cid not in cmap:
        print("존재하지 않는 카테고리ID입니다.")
        return

    if cid in [f"C{str(i).zfill(3)}" for i in range(1, 8)]:
        print("기본 카테고리는 삭제할 수 없습니다.")
        return

    cat_name = cmap[cid]
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6 and f[3] == cat_name:
            print("해당 카테고리를 사용하는 상품이 존재하여 삭제할 수 없습니다. 해당 상품의 카테고리를 먼저 변경해 주세요.")
            return

    updated_cat = [line for line in lines if parse_record(line)[0] != cid]
    write_file(CATEGORY_DATA, updated_cat)
    print("카테고리가 성공적으로 삭제되었습니다.")

# ══════════════════════════════════════════════════════════════
#  SECTION 6 : 쿠폰 관리 및 고객 갱신 함수 (2.3절, 2.4절)
# ══════════════════════════════════════════════════════════════
def next_coupon_id() -> str:
    used_ids = set()
    for line in read_file(COUPON_DATA):
        f = parse_record(line)
        if f and f[0].isdigit():
            used_ids.add(f[0])

    i = 1
    while True:
        if str(i) not in used_ids:
            return str(i)
        i += 1

def create_coupon(coupon_type: str, discount_value: int) -> str | None:
    if coupon_type not in ("RATE", "FLAT"):
        return None
    if coupon_type == "RATE" and not (1 <= discount_value <= 100):
        return None
    if coupon_type == "FLAT" and discount_value < 1:
        return None

    cid = next_coupon_id()
    due_datetime = (current_datetime + timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    try:
        append_line(COUPON_DATA, f"{cid}|{coupon_type}|{discount_value}|{due_datetime}|USABLE")
    except OSError:
        print("데이터 파일 접근에 실패하여 등급 갱신을 중단합니다.")
        sys.exit(1)
    return cid

def issue_grade_coupons(customer_id: str, grade: str) -> None:
    if not GRADE_RE.match(grade):
        return

    new_cids = []
    if grade == "BRONZE":
        new_cids.append(create_coupon("FLAT", 1000))
    elif grade == "SILVER":
        new_cids.append(create_coupon("FLAT", 2000))
        new_cids.append(create_coupon("RATE", 5))
    elif grade == "GOLD":
        new_cids.append(create_coupon("FLAT", 5000))
        new_cids.append(create_coupon("RATE", 10))
        new_cids.append(create_coupon("FLAT", 1000))
        new_cids.append(create_coupon("FLAT", 1000))
    elif grade == "VIP":
        new_cids.append(create_coupon("FLAT", 10000))
        new_cids.append(create_coupon("RATE", 15))
        new_cids.append(create_coupon("FLAT", 2000))
        new_cids.append(create_coupon("FLAT", 2000))
        new_cids.append(create_coupon("FLAT", 2000))

    new_cids = [c for c in new_cids if c]

    updated_lines = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) == 6 and f[0] == customer_id:
            existing = [c.strip() for c in f[5].split(",")] if f[5] else []
            existing.extend(new_cids)
            f[5] = ",".join(existing)
            updated_lines.append("|".join(f))
        else:
            updated_lines.append(line)

    try:
        write_file(CUSTOMER_DATA, updated_lines)
    except OSError:
        print("데이터 파일 접근에 실패하여 등급 갱신을 중단합니다.")
        sys.exit(1)

def update_customer_grades(now: datetime) -> None:
    if now is None: return

    last_y = now.year - 1 if now.month == 1 else now.year
    last_m = 12 if now.month == 1 else now.month - 1

    purchase_totals = {}
    for line in read_file(CUSTOMER_LOG):
        f = parse_record(line)
        if len(f) == 8:
            dt = parse_dt(f[7])
            if dt.year == last_y and dt.month == last_m:
                purchase_totals[f[1]] = purchase_totals.get(f[1], 0) + int(f[3])

    updated_count = 0
    lines = read_file(CUSTOMER_DATA)
    updated_lines = []

    for line in lines:
        f = parse_record(line)
        if len(f) == 6:
            last_up = parse_dt(f[4])
            if now.year != last_up.year or now.month != last_up.month:
                cid = f[0]
                total = purchase_totals.get(cid, 0)

                if total < 30000: new_grade = "BRONZE"
                elif total < 100000: new_grade = "SILVER"
                elif total < 300000: new_grade = "GOLD"
                else: new_grade = "VIP"

                f[3] = new_grade
                f[4] = now.strftime("%Y-%m-%d %H:%M:%S")
                updated_lines.append("|".join(f))

                try:
                    write_file(CUSTOMER_DATA, updated_lines + lines[len(updated_lines):])
                except OSError:
                    print("데이터 파일 접근에 실패하여 등급 갱신을 중단합니다.")
                    sys.exit(1)
                issue_grade_coupons(cid, new_grade)

                lines = read_file(CUSTOMER_DATA)
                updated_lines = lines[:len(updated_lines)]
                updated_count += 1
            else:
                updated_lines.append(line)

    if updated_count > 0:
        print(f"{updated_count}명의 고객 등급이 갱신되었습니다.")

def expire_coupons(now: datetime) -> None:
    if now is None: return

    coupons = _load_coupons()
    expired_ids = []

    updated_coupon_lines = []
    for line in read_file(COUPON_DATA):
        f = parse_record(line)
        if len(f) == 5:
            cid, _, _, cdur, cstat = f
            if cstat == "USABLE" and parse_dt(cdur) < now:
                f[4] = "USED"
                expired_ids.append(cid)
            updated_coupon_lines.append("|".join(f))

    if not expired_ids:
        return

    updated_customer_lines = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) == 6:
            if f[5]:
                c_list = [c.strip() for c in f[5].split(",") if c.strip() not in expired_ids]
                f[5] = ",".join(c_list)
            updated_customer_lines.append("|".join(f))

    try:
        write_file(COUPON_DATA, updated_coupon_lines)
        write_file(CUSTOMER_DATA, updated_customer_lines)
    except OSError:
        print("데이터 파일 접근에 실패하여 쿠폰 만료 처리를 중단합니다.")
        sys.exit(1)

    print(f"총 {len(expired_ids)}개의 쿠폰이 유효기간 만료로 사용 불가 처리되었습니다.")

def run_time_based_auto_tasks(now: datetime) -> None:
    auto_dispose(now)
    update_customer_grades(now)
    expire_coupons(now)

def change_current_datetime() -> None:
    global current_datetime
    if current_datetime:
        print(f"현재 설정된 일시: {current_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    new_dt = input_current_datetime()
    current_datetime = new_dt
    run_time_based_auto_tasks(current_datetime)

def _load_coupons() -> dict:
    coupons = {}
    for line in read_file(COUPON_DATA):
        f = parse_record(line)
        if len(f) == 5:
            coupons[f[0]] = {
                "type": f[1],
                "val": int(f[2]),
                "dur": f[3],
                "stat": f[4]
            }
    return coupons

def get_valid_coupons(customer_id: str) -> list[dict]:
    cmap = _load_coupons()
    valid = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) == 6 and f[0] == customer_id:
            cids = [c.strip() for c in f[5].split(",")] if f[5] else []
            for cid in cids:
                c = cmap.get(cid)
                if c and c["stat"] == "USABLE" and parse_dt(c["dur"]) >= current_datetime:
                    c["id"] = cid
                    valid.append(c)
            break
    return valid

def customer_exists(customer_id: str) -> bool:
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) == 6 and f[0] == customer_id:
            return True
    return False

def show_customer_coupons(customer_id: str) -> None:
    if not customer_exists(customer_id):
        print("등록되지 않은 고객ID입니다.")
        return

    c_list = get_valid_coupons(customer_id)
    if not c_list:
        print("보유 중인 쿠폰이 없습니다.")
        return

    c_list.sort(key=lambda x: int(x["id"]))
    print(f"\n{'쿠폰ID':<8} {'쿠폰 종류':<10} {'할인값':<8} {'유효기간':<22} {'사용 가능 여부'}")
    print("-" * 65)
    for c in c_list:
        print(f"{c['id']:<8} {c['type']:<10} {c['val']:<8} {c['dur']:<22} {c['stat']}")
    print(f"\n총 보유 쿠폰 수: {len(c_list)}개")

def check_coupon(coupon_id: str, c_list: list) -> bool:
    if any(c["id"] == coupon_id for c in c_list):
        return True
    return False

def apply_coupon(c_price: int, coupon_id: str) -> int:
    cmap = _load_coupons()
    c = cmap.get(coupon_id)
    if not c: return c_price

    if c["type"] == "RATE":
        return (c_price * (100 - c["val"])) // 100
    else:
        return max(0, c_price - c["val"])

def mark_coupon_as_used(coupon_id: str) -> None:
    lines = read_file(COUPON_DATA)
    updated_lines = []
    for line in lines:
        f = parse_record(line)
        if len(f) == 5 and f[0] == coupon_id:
            f[4] = "USED"
            updated_lines.append("|".join(f))
        else:
            updated_lines.append(line)
    write_file(COUPON_DATA, updated_lines)

    c_lines = read_file(CUSTOMER_DATA)
    updated_c_lines = []
    for line in c_lines:
        f = parse_record(line)
        if len(f) == 6 and f[5]:
            c_list = [c.strip() for c in f[5].split(",")]
            if coupon_id in c_list:
                c_list.remove(coupon_id)
                f[5] = ",".join(c_list)
            updated_c_lines.append("|".join(f))
        else:
            updated_c_lines.append(line)
    write_file(CUSTOMER_DATA, updated_c_lines)

# ══════════════════════════════════════════════════════════════
#  SECTION 7 : 초기 설정 (6.1절)
# ══════════════════════════════════════════════════════════════
def startup_banner() -> None:
    print("=" * 60)
    print("'스마트 무인 편의점' 재고 관리 프로그램 v1.0")
    print("=" * 60)

def check_files() -> None:
    for f in FILES:
        if not os.path.exists(f):
            print("필수 데이터 파일이 존재하지 않습니다.")
            sys.exit(1)
        if not os.access(f, os.R_OK | os.W_OK):
            print("파일 접근 권한이 없습니다.")
            sys.exit(1)

def init_default_data() -> None:
    lines = read_file(CATEGORY_DATA)
    if not lines:
        default_categories = [
            "C001|냉동식품",
            "C002|냉장식품",
            "C003|과자",
            "C004|음료",
            "C005|주류",
            "C006|생활용품",
            "C007|기타"
        ]
        write_file(CATEGORY_DATA, default_categories)

def input_current_datetime() -> datetime:
    last = get_last_reference_time()
    while True:
        print()
        s = input("현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):\n> ").strip()
        print()
        if not is_valid_datetime(s):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-04-02 14:00:00)")
            continue
        dt = parse_dt(s)
        if last and dt <= last:
            print("현재 일시는 기존 로그의 마지막 시각보다 늦어야 합니다.")
            continue
        print("기준 시각이 설정되었습니다.")
        return dt

def auto_dispose(now: datetime) -> None:
    expired = []
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6 and parse_dt(f[5]) < now:
            pid = f[0]
            stock = calc_stock(pid)
            if stock >= 1:
                expired.append((pid, stock))

    if not expired:
        print("현재 폐기 처리할 유통기한 경과 상품이 없습니다.")
        return

    eid = next_event_id()
    ids_list = []
    for pid, stock in expired:
        ids_list.extend([pid] * stock)

    ids_field = ", ".join(ids_list)
    total_qty = len(ids_list)
    time_s = now.strftime("%Y-%m-%d %H:%M:%S")
    record = f"{eid}|{ids_field}|DISPOSE|{time_s}"
    try:
        append_line(PRODUCT_LOG, record)
    except OSError:
        print("데이터 파일 접근에 실패하여 폐기 처리를 중단합니다.")
        sys.exit(1)
    print(f"총 {total_qty}개의 재고가 유통기한 경과로 자동 폐기 처리되었습니다.")

# ══════════════════════════════════════════════════════════════
#  SECTION 8 : 메인 메뉴 (6.2절)
# ══════════════════════════════════════════════════════════════
def main_menu() -> None:
    global current_datetime
    while True:
        print(f"\n현재 일시: {current_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
        print("1. 고객 관리")
        print("2. 재고 관리")
        print("3. 판매 처리")
        print("4. 가상의 일시 기준 재고 조회")
        print("5. 현재 일시 변경")
        print("6. 카테고리 관리")
        print("0. 종료")
        choice = input("메뉴 > ").strip()

        if not re.fullmatch(r"[0-9]+", choice):
            print("숫자를 입력해주세요.")
            continue

        match choice:
            case "1":
                menu_customer()
            case "2":
                menu_inventory()
            case "3":
                menu_sales()
            case "4":
                menu_virtual_stock()
            case "5":
                change_current_datetime()
            case "6":
                manage_category_menu()
            case "0":
                print("프로그램을 종료합니다.")
                sys.exit(0)
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

# ══════════════════════════════════════════════════════════════
#  SECTION 9 : 고객 관리 (6.3절)
# ══════════════════════════════════════════════════════════════
def menu_customer() -> bool:
    while True:
        print("\n1. 고객 등록\n2. 고객 조회\n3. 쿠폰 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()

        if not re.fullmatch(r"[0-9]+", choice):
            print("숫자를 입력해주세요.")
            continue

        match choice:
            case "1":
                register_customer()
                return True
            case "2":
                search_customer()
                return True
            case "3":
                while True:
                    cid = input("고객ID: ").strip()
                    if not is_valid_customer_id(cid):
                        print("고객ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
                        continue
                    show_customer_coupons(cid)
                    return True
            case "0":
                return False
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

def register_customer() -> None:
    while True:
        name = input("고객명: ").strip()
        if not is_valid_customer_name(name):
            print("고객명은 한글 2~5자 또는 영문 3~20자여야 합니다.")
            continue
        break
    new_id = next_customer_id()
    if new_id is None:
        print("사용 가능한 고객ID가 없어 고객 등록을 진행할 수 없습니다.")
        return
    print(f"\n고객ID: {new_id}  고객명: {name}  초기 포인트: 0")

    while True:
        confirm = input("등록하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            now_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")
            append_line(CUSTOMER_DATA, f"{new_id}|{name}|0|BRONZE|{now_s}|")
            print("고객 등록이 완료되었습니다.")
            return
        elif confirm in ("N", "n"):
            print("등록을 취소했습니다.")
            return
        else:
            print("Y 또는 N으로 입력해 주세요.")

def search_customer() -> None:
    while True:
        raw = input("검색 (예: 1 1023 또는 2 김철수): ").strip()

        if " " not in raw:
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        if raw.startswith(" ") or raw.endswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        parts = raw.split(" ", 1)

        if parts[0] not in ("1", "2") or parts[1].startswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        criterion, keyword = parts[0], parts[1].strip()
        is_valid = True

        if criterion == "1":
            if not is_valid_customer_id(keyword):
                print("고객ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
                is_valid = False
        elif criterion == "2":
            if not is_valid_customer_name(keyword):
                print("고객명은 한글 2~5자 또는 영문 3~20자여야 합니다.")
                is_valid = False

        if not is_valid:
            continue

        break

    results = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) == 6:
            if criterion == "1" and f[0] == keyword:
                results.append(f)
            elif criterion == "2" and f[1] == keyword:
                results.append(f)

    if not results:
        print("검색 결과가 없습니다.")
        return
    results.sort(key=lambda x: x[0])
    print(f"\n{'고객ID':<8} {'고객명':<12} {'포인트':<8} {'고객등급':<10} {'갱신 시각':<22} {'쿠폰ID'}")
    print("-" * 75)
    for f in results:
        print(f"{f[0]:<8} {f[1]:<12} {f[2]:<8} {f[3]:<10} {f[4]:<22} {f[5]}")
    print(f"\n총 {len(results)}건 검색됨")

# ══════════════════════════════════════════════════════════════
#  SECTION 10 : 재고 관리 (6.4절)
# ══════════════════════════════════════════════════════════════
def menu_inventory() -> bool:
    while True:
        print("\n1. 상품 입고\n2. 재고 조회\n3. 전체 재고 출력\n4. 폐기 상품 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()
        if not re.fullmatch(r"[0-9]+", choice):
            print("숫자를 입력해주세요.")
            continue
        match choice:
            case "1":
                receive_product()
                return True
            case "2":
                search_inventory()
                return True
            case "3":
                print_all_inventory()
                return True
            case "4":
                search_disposed()
                return True
            case "0":
                return False
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

def receive_product() -> None:
    while True:
        name = input("상품명: ").strip()
        if len(name) < 1 or "\\" in name or "/" in name or "|" in name:
            print("상품명은 1글자 이상이어야 하며 '\\', '/', '|'를 포함할 수 없습니다.")
            continue
        break
    while True:
        cat = input("카테고리: ").strip()
        if not is_valid_category(cat):
            print("정확한 카테고리 문자열 중 하나를 입력해 주세요.")
            continue
        break
    while True:
        price_s = input("상품 가격: ").strip()
        if not is_valid_price(price_s):
            print("상품 가격은 0 이상의 정수여야 합니다.")
            continue
        break
    while True:
        in_date = input("입고일 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(in_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-03-25 14:00:00)")
            continue
        if parse_dt(in_date) > current_datetime:
            print("입고일은 현재 일시 이전이어야 합니다.")
            continue
        break
    while True:
        exp_date = input("유통기한 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(exp_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-06-30 23:59:59)")
            continue
        if parse_dt(exp_date) <= parse_dt(in_date):
            print("유통기한은 입고일 이후여야 합니다.")
            continue
        break

    new_pid = next_product_id()
    if new_pid is None:
        print("사용 가능한 상품ID가 없어 상품 입고를 진행할 수 없습니다.")
        return
    qty = 1

    print(f"""
    상품ID: {new_pid}
    상품명: {name}
    카테고리: {cat}
    가격: {price_s}
    입고일: {in_date}
    유통기한: {exp_date}
    """)

    while True:
        confirm = input("입고하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            append_line(PRODUCT_DATA, f"{new_pid}|{name}|{price_s}|{cat}|{in_date}|{exp_date}")
            eid = next_event_id()
            time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")
            ids = [new_pid] * qty
            ids_field = ", ".join(ids)

            if not validate_event_time(ids, current_datetime):
                return

            append_line(PRODUCT_LOG, f"{eid}|{ids_field}|IN|{time_s}")
            print("상품 입고가 완료되었습니다.")
            return

        elif confirm in ("N", "n"):
            print("입고를 취소했습니다.")
            return
        else:
            print("Y 또는 N으로 입력해 주세요.")

def _load_products() -> dict:
    products = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            products[f[0]] = {
                "name": f[1], "price": int(f[2]),
                "category": f[3], "in_date": f[4], "exp_date": f[5]
            }
    return products

def search_inventory() -> None:
    criteria_map = {"1": "상품ID", "2": "상품명", "3": "상품 가격",
                    "4": "카테고리", "5": "입고일", "6": "유통기한"}
    while True:
        raw = input("검색 기준번호 검색어 (예: 2 콜라): ").strip()

        if " " not in raw:
            print("형식에 맞게 입력해 주세요. (예: 2 콜라)")
            continue

        parts = raw.split(" ", 1)

        if parts[0] not in criteria_map or parts[1].startswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 2 콜라)")
            continue

        crit, keyword = parts[0], parts[1].strip()
        is_valid = True

        if crit == "1":
            if not is_valid_product_id(keyword):
                print("상품ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
                is_valid = False
        elif crit == "2":
            if len(keyword) < 1 or "\\" in keyword or "/" in keyword or "|" in keyword:
                print("상품명은 1글자 이상이어야 하며 '\\', '/', '|'를 포함할 수 없습니다.")
                is_valid = False
        elif crit == "3":
            test_kw = keyword.replace("~", "").replace("이하", "").replace("이상", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("상품 가격은 0 이상의 정수여야 합니다.")
                is_valid = False
        elif crit == "4":
            if not is_valid_category(keyword):
                print("정확한 카테고리 문자열 중 하나를 입력해 주세요.")
                is_valid = False
        elif crit == "5":
            test_kw = keyword.replace("~", "").replace("-", "").replace(":", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("날짜 검색 형식이 올바르지 않습니다.")
                is_valid = False
        elif crit == "6":
            test_kw = keyword.replace("~", "").replace("-", "").replace(":", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("유통기한 검색 형식이 올바르지 않습니다.")
                is_valid = False

        if not is_valid:
            continue

        break

    products = _load_products()
    stocks = get_all_stocks()
    matched = []
    for pid, p in products.items():
        if stocks.get(pid, 0) < 1:
            continue
        match crit:
            case "1":
                hit = pid == keyword
            case "2":
                hit = keyword in p["name"]
            case "3":
                hit = _price_filter(keyword, p["price"])
            case "4":
                hit = p["category"] == keyword
            case "5":
                hit = _date_range_filter(keyword, p["in_date"])
            case "6":
                hit = _date_range_filter(keyword, p["exp_date"])
            case _:
                hit = False
        if hit:
            matched.append((pid, p, stocks[pid]))

    if not matched:
        print("검색 결과가 없습니다.")
        return

    matched.sort(key=lambda x: x[0])
    _print_inventory_table(matched)

def _price_filter(keyword: str, price: int) -> bool:
    keyword = keyword.replace(" ", "")
    if "~" in keyword:
        parts = keyword.split("~", 1)
        lo = parts[0]
        hi = parts[1]
        lo_val = int(lo) if lo.isdigit() else 0
        hi_val = int(hi) if hi.isdigit() else float('inf')
        return lo_val <= price <= hi_val
    if keyword.endswith("이하") and keyword[:-2].isdigit():
        return price <= int(keyword[:-2])
    if keyword.endswith("이상") and keyword[:-2].isdigit():
        return price >= int(keyword[:-2])
    if keyword.isdigit():
        return str(price) == keyword
    return False

def parse_date_for_search(d_str: str, is_end: bool) -> str:
    d_str = d_str.strip()
    if not d_str:
        return "9999-12-31 23:59:59" if is_end else "0000-00-00 00:00:00"
    if len(d_str) == 4:
        return f"{d_str}-12-31 23:59:59" if is_end else f"{d_str}-01-01 00:00:00"
    if len(d_str) == 7:
        if is_end:
            try:
                y, m = int(d_str[:4]), int(d_str[5:7])
                last_day = calendar.monthrange(y, m)[1]
                return f"{d_str}-{last_day} 23:59:59"
            except ValueError:
                return f"{d_str}-31 23:59:59"
        else:
            return f"{d_str}-01 00:00:00"
    if len(d_str) == 10:
        return f"{d_str} 23:59:59" if is_end else f"{d_str} 00:00:00"
    if len(d_str) == 13:
        return f"{d_str}:59:59" if is_end else f"{d_str}:00:00"
    if len(d_str) == 16:
        return f"{d_str}:59" if is_end else f"{d_str}:00"
    return d_str

def _date_range_filter(keyword: str, date_s: str) -> bool:
    if "~" in keyword:
        parts = keyword.split("~", 1)
        lo = parse_date_for_search(parts[0], False)
        hi = parse_date_for_search(parts[1], True)
        return lo <= date_s <= hi
    else:
        lo = parse_date_for_search(keyword, False)
        hi = parse_date_for_search(keyword, True)
        return lo <= date_s <= hi

def _print_inventory_table(items: list) -> None:
    print(f"\n{'상품ID':<8}{'상품명':<16}{'카테고리':<12}{'가격':>8}{'수량':>6}  {'입고일':<22}{'유통기한'}")
    print("-" * 90)
    for pid, p, qty in items:
        print(f"{pid:<8}{p['name']:<16}{p['category']:<12}{p['price']:>8}{qty:>6}"
              f"  {p['in_date']:<22}{p['exp_date']}")
    print(f"\n총 {len(items)}개 상품")

def print_all_inventory() -> None:
    products = _load_products()
    stocks = get_all_stocks()
    in_stock = [(pid, p, stocks.get(pid, 0)) for pid, p in products.items()
                if stocks.get(pid, 0) >= 1]
    if not in_stock:
        print("등록된 상품이 없습니다.")
        return
    in_stock.sort(key=lambda x: x[0])
    _print_inventory_table(in_stock)
    print(f"총 {len(in_stock)}종  총 수량 {sum(q for _, _, q in in_stock)}개")

def search_disposed() -> None:
    products = _load_products()
    disposed = []
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        if len(f) == 4 and f[2] == "DISPOSE":
            disposed.append(f)
    if not disposed:
        print("폐기 처리된 상품 내역이 존재하지 않습니다.")
        return
    disposed.sort(key=lambda x: x[3], reverse=True)
    print(f"\n{'이벤트ID':<12}{'상품명':<30}{'폐기수량':>8}  {'이벤트 시각'}")
    print("-" * 72)
    for f in disposed:
        ids = [pid.strip() for pid in f[1].split(",") if pid.strip()]
        names = ", ".join(products.get(pid, {}).get("name", "?") for pid in ids)
        print(f"{f[0]:<12}{names:<30}{len(ids):>8}  {f[3]}")

# ══════════════════════════════════════════════════════════════
#  SECTION 11 : 판매 처리 (6.5절)
# ══════════════════════════════════════════════════════════════
def calculate_discounted_total(product_ids: list[str], product_map: dict, current_time: datetime) -> list[int]:
    discounted_prices = []
    for pid in product_ids:
        p = product_map[pid]
        orig_p = int(p[2])
        e_date = parse_dt(p[5])
        diff = e_date - current_time

        if diff <= timedelta(days=3):
            discounted_prices.append(int(orig_p * 0.5))
        elif diff <= timedelta(days=7):
            discounted_prices.append(int(orig_p * 0.8))
        else:
            discounted_prices.append(orig_p)

    return discounted_prices

def print_payment_confirm_summary(
    product_ids: list[str],
    product_map: dict,
    discounted_prices: list[int],
    customer_id: str,
    customers: dict,
    payment: str,
    applied_coupon_id: str,
    total_price: int,
    earned_points: int = 0,
    used_points: int = 0,
    remain_points: int = 0) -> None:
    print("\n[결제 최종 확인]")
    print(f"{'상품ID':<8} {'상품명':<16} {'정가':>8} {'최종 산정 가격':>14}")
    print("-" * 55)

    for pid, final_price in zip(product_ids, discounted_prices):
        p = product_map[pid]
        print(f"{pid:<8} {p[1]:<16} {int(p[2]):>8} {final_price:>14}")

    if customer_id == "0000":
        print("고객 정보: 비회원")
    else:
        print(f"고객 정보: {customer_id} / {customers[customer_id]['name']}")
        print(f"결제 전 보유 포인트: {customers[customer_id]['points']}")

    print(f"결제 방식: {payment}")
    print(f"쿠폰 적용 여부: {'적용함' if applied_coupon_id else '적용 안함'}")
    print(f"적용 쿠폰ID: {applied_coupon_id if applied_coupon_id else '-'}")
    print(f"최종 합계 금액: {total_price}원")

    if customer_id != "0000":
        if payment == "포인트":
            print(f"차감될 포인트: {used_points}")
            print(f"결제 후 잔여 포인트: {remain_points}")
        else:
            print(f"적립 예정 포인트: {earned_points}")

def menu_sales() -> bool:
    print("\n[판매 처리]")

    product_ids = []
    product_map = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            product_map[f[0]] = f

    while True:
        pid = input("상품ID 입력 (종료: 0): ").strip()

        if pid == "0":
            if not product_ids:
                print("판매할 상품을 한 개 이상 선택해야 합니다.")
                continue
            break

        if not is_valid_product_id(pid):
            print("상품ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
            continue

        if pid not in product_map:
            print("해당 ID의 상품이 존재하지 않습니다.")
            continue

        if pid in product_ids:
            print("같은 상품ID를 중복 입력할 수 없습니다.")
            continue

        stock = calc_stock(pid)
        if stock <= 0:
            print("해당 상품의 재고가 없습니다.")
            continue

        exp_date = parse_dt(product_map[pid][5])
        if exp_date < current_datetime:
            print("유통기한 지난 상품입니다.")
            continue

        product_ids.append(pid)
        p = product_map[pid]

        orig_price = int(p[2])
        discount_price = orig_price
        delta = exp_date - current_datetime

        if delta <= timedelta(days=3):
            discount_price = int(orig_price * 0.5)
            print(f"상품명: {p[1]}, 카테고리: {p[3]}, 상품 가격: {orig_price} -> {discount_price} (50% 자동 할인), 현재 재고 수량: {stock}")
        elif delta <= timedelta(days=7):
            discount_price = int(orig_price * 0.8)
            print(f"상품명: {p[1]}, 카테고리: {p[3]}, 상품 가격: {orig_price} -> {discount_price} (20% 자동 할인), 현재 재고 수량: {stock}")
        else:
            print(f"상품명: {p[1]}, 카테고리: {p[3]}, 상품 가격: {orig_price}, 현재 재고 수량: {stock}")

    quantity = len(product_ids)
    discounted_prices = calculate_discounted_total(product_ids, product_map, current_datetime)

    while True:
        customer_id = input("고객ID 입력 (비회원: 0): ").strip()

        customers = {}
        for line in read_file(CUSTOMER_DATA):
            f = parse_record(line)
            if len(f) == 6:
                customers[f[0]] = {"name": f[1], "points": int(f[2])}

        if customer_id == "0":
            customer_id = "0000"
            break

        if customer_id == "0000":
            print("0000은 비회원 거래 기록용 ID이므로 회원ID로 입력할 수 없습니다.")
            continue

        if not is_valid_customer_id(customer_id):
            print("고객ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
            continue

        if customer_id not in customers:
            print("등록되지 않은 고객ID입니다.")
            continue

        print(f"고객명: {customers[customer_id]['name']}, 보유 포인트: {customers[customer_id]['points']}")
        break

    applied_coupon_id = ""
    if customer_id != "0000":
        c_list = get_valid_coupons(customer_id)
        if c_list:
            show_customer_coupons(customer_id)
            while True:
                cid_in = input("사용할 쿠폰ID (사용 안함: 0): ").strip()
                if cid_in == "0":
                    break
                if not check_coupon(cid_in, c_list):
                    print("유효하지 않거나 유효기간이 만료된 쿠폰ID입니다.")
                    continue

                print("현재 선택된 상품 목록: ", ", ".join(product_ids))
                while True:
                    c_pid = input("쿠폰을 적용할 상품ID: ").strip()
                    if c_pid not in product_ids:
                        print("선택되지 않은 상품ID입니다.")
                        continue
                    break

                idx = product_ids.index(c_pid)
                prev_price = discounted_prices[idx]
                discounted_prices[idx] = apply_coupon(prev_price, cid_in)
                applied_coupon_id = cid_in
                print(f"쿠폰 적용 완료. 할인 후 해당 상품 가격: {discounted_prices[idx]}")
                break

    total_price = sum(discounted_prices)

    while True:
        if customer_id == "0000":
            payment = input("결제 방식 선택 (현금/카드): ").strip()
            if payment not in ("현금", "카드"):
                print("현금, 카드 중 하나를 입력해주세요.")
                continue
        else:
            if applied_coupon_id:
                payment = input("결제 방식 선택 (현금/카드): ").strip()
                if payment not in ("현금", "카드"):
                    print("현금, 카드 중 하나를 입력해 주세요.")
                    continue
            else:
                payment = input("결제 방식 선택 (현금/카드/포인트): ").strip()
                if payment not in ("현금", "카드", "포인트"):
                    print("현금, 카드, 포인트 중 하나를 입력해주세요.")
                    continue

        if payment == "포인트" and customer_id != "0000":
            if customers[customer_id]["points"] < total_price:
                print(f"보유 포인트: {customers[customer_id]['points']}, 결제 필요 금액: {total_price}")
                print("포인트가 부족합니다.")
                continue

        break

    used_points = 0
    remain_points = 0
    earned_points = 0

    if customer_id != "0000":
        current_points = customers[customer_id]["points"]

        if payment == "포인트":
            used_points = total_price
            remain_points = current_points - used_points
        else:
            earned_points = 0 if applied_coupon_id else int(total_price * 0.01)

    print_payment_confirm_summary(
        product_ids=product_ids,
        product_map=product_map,
        discounted_prices=discounted_prices,
        customer_id=customer_id,
        customers=customers,
        payment=payment,
        applied_coupon_id=applied_coupon_id,
        total_price=total_price,
        earned_points=earned_points,
        used_points=used_points,
        remain_points=remain_points,
    )

    while True:
        confirm = input("결제하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y", "N", "n"):
            break
        print("Y 또는 N으로 입력해 주세요.")

    if confirm in ("N", "n"):
        print("결제를 취소했습니다.")
        return True

    if not validate_event_time(product_ids, current_datetime):
        return True

    if customer_id != "0000":
        if payment == "포인트":
            customers[customer_id]["points"] -= used_points
        else:
            customers[customer_id]["points"] += earned_points

        updated_lines = []
        for line in read_file(CUSTOMER_DATA):
            f = parse_record(line)
            if len(f) == 6 and f[0] == customer_id:
                updated_lines.append(f"{f[0]}|{f[1]}|{customers[customer_id]['points']}|{f[3]}|{f[4]}|{f[5]}")
            else:
                updated_lines.append(line)
        write_file(CUSTOMER_DATA, updated_lines)

    if applied_coupon_id:
        mark_coupon_as_used(applied_coupon_id)

    eid = next_event_id()
    ids_field = ", ".join(product_ids)
    time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")

    append_line(PRODUCT_LOG, f"{eid}|{ids_field}|OUT|{time_s}")

    purchase_id = next_purchase_id()
    append_line(
        CUSTOMER_LOG,
        f"{purchase_id}|{customer_id}|{ids_field}|{total_price}|{applied_coupon_id}|{payment}|{quantity}|{time_s}"
    )

    print("결제가 완료되었습니다.")

    if customer_id != "0000":
        print("[포인트 변동 내역]")
        if payment == "포인트":
            print(f"차감 포인트: {used_points}")
            print(f"잔여 포인트: {customers[customer_id]['points']}")
        else:
            print(f"적립 포인트: {earned_points}")
            print(f"잔여 포인트: {customers[customer_id]['points']}")

    return True

# ══════════════════════════════════════════════════════════════
#  SECTION 12 : 가상의 일시 기준 재고 조회 (6.6절)
# ══════════════════════════════════════════════════════════════
def menu_virtual_stock() -> None:
    while True:
        s = input("가상의 일시 입력 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(s):
            print("올바른 날짜 형식이 아닙니다.")
            continue
        virtual_dt = parse_dt(s)
        break
    products = _load_products()
    stocks = get_all_stocks(as_of=virtual_dt)
    available = [
        (pid, p, stocks[pid])
        for pid, p in products.items()
        if stocks.get(pid, 0) >= 1 and parse_dt(p["exp_date"]) >= virtual_dt
    ]
    available.sort(key=lambda x: x[0])
    print("\n[판매 가능한 재고 목록]")
    print(f"{'상품ID':<8} {'상품명':<20} {'수량'}")
    print("-" * 36)
    for pid, p, qty in available:
        print(f"{pid:<8} {p['name']:<20} {qty}")
    print(f"\n총 상품 개수: {len(available)}개")

# ══════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════
def main():
    global current_datetime
    startup_banner()
    check_files()
    init_default_data()
    check_integrity()
    current_datetime = input_current_datetime()
    run_time_based_auto_tasks(current_datetime)
    main_menu()

if __name__ == "__main__":
    main()

'스마트 무인 편의점' 재고 관리 프로그램 v1.0
필수 파일이 존재하지 않습니다.


SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
